In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, t_raman_pulse, image_detuning, t_ramp_2, v_squeeze, phase_slm):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)
                
                self.p.t_continuous_rabi = 2000.e-6

                self.p.v_pd_hf_tweezer_squeeze_power = {v_squeeze}
                self.p.t_tweezer_squeezer_ramp_2 = {t_ramp_2}

                # self.xvar('t_raman_pulse', [0.,self.p.t_raman_pi_pulse / 2])
                # self.p.t_raman_pulse = 0.
                self.p.t_raman_pulse = {t_raman_pulse}
                # self.p.t_raman_pulse = self.p.t_raman_pi_pulse
                
                self.xvar('amp_imaging', np.linspace(.2,2., 5))
                self.p.amp_imaging = 1.2
                
                # self.xvar('dimension_slm_mask',np.linspace(15.e-6,250.e-6,10))
                self.p.dimension_slm_mask = 20.e-6

                # self.xvar('phase_slm_mask',np.linspace(0.01*np.pi,.4*np.pi,5))
                self.p.phase_slm_mask = {phase_slm}

                self.p.frequency_detuned_hf_midpoint = {image_detuning}

                self.p.t_tweezer_hold = 15.e-3
                self.p.t_tof = 20.e-6
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True,cubic_ramp_squeeze=False)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                if self.p.t_raman_pulse < 1.e-6:
                    delay(self.p.t_raman_pi_pulse)
                else:
                    self.raman.pulse(t=self.p.t_raman_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(self.p.t_continuous_rabi)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [3]:
v_squeeze = np.array([.246,.985,1.97,3.94])

image_detuning = ([-511.e6,-510.e6,-503.e6,-490.e6])

phase_slm = np.array([.15 * np.pi, .13 * np.pi, .13 * np.pi, .13 * np.pi])

t_ramp_2 = np.array([11.e-3, 17.e-3, 17.e-3, 24.e-3])

t_raman_pulse = np.array([0.e-6,8.8210e-06 / 2, 8.8210e-06])

for i in range(len(v_squeeze)):
       for t in range(len(t_raman_pulse)):
              print(v_squeeze[i], t_raman_pulse[t])
              eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(v_squeeze=v_squeeze[i], t_raman_pulse=t_raman_pulse[t], image_detuning=image_detuning[i], phase_slm=phase_slm[i], t_ramp_2=t_ramp_2[i]))
              eBuilder.run_expt()

0.246 0.0
0  50 values of amp_imaging. 50 total shots. 150 total images expected.
Run ID: 75833
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 818], 'phase': 0.15, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.15 pi, x-center = 993, y-center = 818

 Run ID: 75833

Sent: {'mask': 'spot', 'center': [993, 818], 'phase': 0.15, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.15 pi, x-center = 993, y-center = 818

shot 1/50 done

Sent: {'mask': 'spot', 'center': [993, 818], 'phase': 0.15, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.15 pi, x-center = 993, y-center = 818

shot 2/50 done

Sent: {'mask': 'spot', 'center': [993, 818], 'phase': 0.15, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.15 pi, 

In [4]:
from kexp import EthernetRelay
from waxx.util.guis.als.als_gui_client import ALSGuiClient
from waxx.util.guis.precilaser.precilaser_gui_client import PrecilaserGuiClient

relay = EthernetRelay()
relay.source_off()

als = ALSGuiClient()
ok = als.run_shutdown_sequence()

precilaser = PrecilaserGuiClient()
ok = precilaser.run_shutdown_sequence()